<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 01 preparation · Inspect the fixed sources</h1><p>This is the source-inspection part, not the complete Bronze lab. The complete lab also requires real Delta writes and replay evidence.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>تحضير اللاب 01 · فحص المصادر الثابتة</h1><p>هذا جزء فحص المصادر لا لاب Bronze كاملًا؛ فاللاب الكامل يتطلب كتابة Delta فعلية وأدلة الإعادة.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Goal and setup</h2><p>Read the manifest, count the three base feeds and examine keys, relationships and city-label formatting. Keep the whole repository together; shared code is in <code>src/masar/sources.py</code>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الهدف والإعداد</h2><p>اقرأ السجل وعد المصادر الثلاثة وافحص المفاتيح والعلاقات وكتابة المدن. احتفظ بالمستودع كاملًا؛ الكود المشترك في <code dir="ltr">src/masar/sources.py</code>.</p></td></tr></tbody></table>

In [1]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Scope: preparatory Python helper only; Spark/Delta are not executed.")

Scope: preparatory Python helper only; Spark/Delta are not executed.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Verify and load</h2><p>Reject changed source files before interpreting their content. This does not modify the inputs.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>1. التحقق والتحميل</h2><p>ارفض تغير ملفات المصدر قبل تفسير محتواها. لا تغير الخطوة المدخلات.</p></td></tr></tbody></table>

In [2]:
from masar.sources import verify_manifest, load_sources, profile_sources
SOURCE = ROOT / "data" / "masar-small-v1"
manifest = verify_manifest(SOURCE)
feeds = load_sources(SOURCE)
print("Dataset:", manifest["label"])
print("Verified manifest files:", len(manifest["files"]))
print(json.dumps({name: len(rows) for name, rows in feeds.items()}, indent=2))
print("Trip columns:", list(feeds["trips"][0]))
print("First synthetic event:", json.dumps(feeds["gps_events"][0], ensure_ascii=False))

Dataset: MASAR_SMALL_V1
Verified manifest files: 10
{
  "trips": 72,
  "drivers": 6,
  "gps_events": 216
}
Trip columns: ['trip_id', 'driver_id', 'city', 'start_ts', 'end_ts', 'fare_sar', 'distance_km']
First synthetic event: {"city": "Riyadh", "event_id": "SYN_E0001_0", "event_ts": "2026-06-01T06:00:00+03:00", "location": {"lat": 24.7, "lon": 46.7}, "synthetic": true, "trip_id": "SYN_T0001"}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2. Profile before cleaning</h2><p>Count keys and inspect relationships. The city-normalization comparison is a view of the inputs, not a rewrite of Bronze.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>2. افحص قبل التنظيف</h2><p>عد المفاتيح وافحص العلاقات. مقارنة توحيد المدن فحص للمدخلات، وليست إعادة كتابة Bronze.</p></td></tr></tbody></table>

In [3]:
result = profile_sources(SOURCE)
print(json.dumps(result["profile"], indent=2))
print(json.dumps(result["relations"], indent=2))
print(json.dumps(result["city_profile"], indent=2))

{
  "trips": {
    "rows": 72,
    "key": "trip_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "distance_km": 0,
      "driver_id": 0,
      "end_ts": 0,
      "fare_sar": 0,
      "start_ts": 0,
      "trip_id": 0
    }
  },
  "drivers": {
    "rows": 6,
    "key": "driver_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "driver_id": 0,
      "driver_rating": 0,
      "vehicle_type": 0
    }
  },
  "gps_events": {
    "rows": 216,
    "key": "event_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "event_id": 0,
      "event_ts": 0,
      "location": 0,
      "synthetic": 0,
      "trip_id": 0
    }
  }
}
{
  "trips_without_driver": 0,
  "events_without_trip": 0,
  "events_with_invalid_coordinates": 0
}
{
  "original_labels": {
    " dammam ": 3,
    " jeddah ": 3,
    " riyad

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3. Check and preserve evidence</h2><p>The checks below concern the source files only. A replay into append-only Bronze is a separate operation and has not been run here.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>3. تحقق واحفظ الأدلة</h2><p>تخص الفحوص التالية ملفات المصدر فقط. إعادة الاستيعاب في Bronze بنمط الإضافة عملية منفصلة لم تنفذ هنا.</p></td></tr></tbody></table>

In [4]:
if not all(result["checks"].values()):
    raise AssertionError(result["checks"])
output = RUN / "source_inspection.json"
output.write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print(json.dumps(result["checks"], indent=2))
print("PASS: source inspection only")
print("Saved:", output.name)
print("Real Delta lab: NOT VERIFIED by this notebook")

{
  "source_counts": true,
  "unique_base_keys": true,
  "base_top_level_complete": true,
  "valid_links_and_coordinates": true,
  "three_events_per_trip": true,
  "base_city_set": true
}
PASS: source inspection only
Saved: source_inspection.json
Real Delta lab: NOT VERIFIED by this notebook


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Interpretation and next step</h2><p>Explain why joining each trip to all its events changes the row grain. Explain why a unique base key does not guarantee uniqueness after replay. Return to the Lab 01 contract; its engine steps remain pending.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>التفسير والخطوة التالية</h2><p>اشرح لماذا يغير ربط الرحلة بجميع أحداثها مستوى الصف، ولماذا لا يضمن المفتاح الفريد في المصدر عدم التكرار بعد الإعادة. ارجع إلى مواصفات اللاب 01؛ خطوات محركه ما تزال معلقة.</p></td></tr></tbody></table>